# Task 1 — SE(3)-Equivariant Flow Matching for SGS Closure
**Owner: Prerona Mitra** · EPITA DSA 2026

Investigates whether framing SGS stress prediction as a *generative* problem (flow matching)
outperforms a well-designed deterministic equivariant model.  Four variants:

| Variant | Description |
|---------|-------------|
| V1 | SE(3)-equivariant CFM — e3nn EGNN + torchdiffeq (proposed) |
| V2 | SE(3)-equivariant deterministic regression — same backbone, MSE loss |
| V3 | Non-equivariant CFM — MLP + 24-rotation augmentation |
| V4 | Inference-only — V1 vs all Task 2 baselines (unified table) |

## 0. Setup

In [ ]:
# Install / upgrade key packages (Kaggle A100 — run once)
# !pip install -q e3nn torchdiffeq torch-geometric wandb

import sys, os
from pathlib import Path

# Point Python at the repo root
REPO = Path('.').resolve().parent   # adjust if running from elsewhere
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import torch
import numpy as np
import matplotlib.pyplot as plt

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 1. Data Loading

In [ ]:
from data.jhtdb import generate_synthetic_les_data, JHTDBLoader
from data.dataset import build_dataloaders

USE_JHTDB = False          # set True + provide JHTDB_TOKEN env var for real data
JHTDB_TOKEN = os.environ.get('JHTDB_TOKEN', '')

if USE_JHTDB and JHTDB_TOKEN:
    loader = JHTDBLoader(token=JHTDB_TOKEN, cache_dir='../jhtdb_cache')
    tau_field, grad_field = loader.prepare_les_data(time_idx=0)
else:
    print('Using synthetic LES data (Kolmogorov spectrum, Smagorinsky ground truth).')
    tau_field, grad_field = generate_synthetic_les_data(n_les=64, seed=0)

print(f'tau_field:  {tau_field.shape}  dtype={tau_field.dtype}')
print(f'grad_field: {grad_field.shape}')

train_loader, val_loader, test_loader, stats = build_dataloaders(
    tau_field, grad_field,
    n_train=4000, n_val=500, n_test=500,
    patch_size=8, k_neighbours=26,
    batch_size=16, num_workers=0, seed=42,
)
print(f'Train batches: {len(train_loader)},  Val: {len(val_loader)},  Test: {len(test_loader)}')
torch.save(stats, '../runs/stats_shared.pt')
print('Normalisation stats saved.')

## 2. Model Instantiation

In [ ]:
from models.flow_matching import build_model
import yaml

with open('../configs/v1.yaml') as f: cfg_v1 = yaml.safe_load(f)
with open('../configs/v2.yaml') as f: cfg_v2 = yaml.safe_load(f)
with open('../configs/v3.yaml') as f: cfg_v3 = yaml.safe_load(f)

model_v1 = build_model('v1', cfg_v1['model'])
model_v2 = build_model('v2', cfg_v2['model'])
model_v3 = build_model('v3', cfg_v3['model'])

def nparams(m): return sum(p.numel() for p in m.parameters())
print(f'V1 (SE3-CFM):          {nparams(model_v1):>9,} params')
print(f'V2 (SE3-Regression):   {nparams(model_v2):>9,} params')
print(f'V3 (MLP-CFM):          {nparams(model_v3):>9,} params')

## 3. Training — V1: SE(3)-Equivariant Conditional Flow Matching

In [ ]:
from training.trainer import Trainer

trainer_v1 = Trainer(
    model=model_v1, variant='v1',
    train_loader=train_loader, val_loader=val_loader,
    cfg=cfg_v1['training'], device=DEVICE, out_dir='../runs/v1',
)
trainer_v1.train()
torch.save(stats, '../runs/v1/stats.pt')

In [ ]:
# Plot training curve
h = trainer_v1.history
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(h['train_loss'], alpha=0.6, label='Train loss')
steps, vals = zip(*h['val_loss'])
ax.plot(steps, vals, 'o-', label='Val loss')
ax.set_xlabel('Step'); ax.set_ylabel('CFM loss'); ax.set_title('V1 Training')
ax.legend(); ax.set_yscale('log')
plt.tight_layout()
plt.savefig('../figures/v1_training_curve.png', dpi=150)
plt.show()

## 4. Training — V2: SE(3)-Equivariant Deterministic Regression

In [ ]:
trainer_v2 = Trainer(
    model=model_v2, variant='v2',
    train_loader=train_loader, val_loader=val_loader,
    cfg=cfg_v2['training'], device=DEVICE, out_dir='../runs/v2',
)
trainer_v2.train()
torch.save(stats, '../runs/v2/stats.pt')

## 5. Training — V3: Non-Equivariant CFM with Rotation Augmentation

In [ ]:
trainer_v3 = Trainer(
    model=model_v3, variant='v3',
    train_loader=train_loader, val_loader=val_loader,
    cfg=cfg_v3['training'], device=DEVICE, out_dir='../runs/v3',
)
trainer_v3.train()
torch.save(stats, '../runs/v3/stats.pt')

## 6. Evaluation — V1, V2, V3

In [ ]:
from evaluation.metrics import compute_all_metrics, save_results_csv

RESULTS_CSV = Path('../results/baselines_results.csv')
RESULTS_CSV.parent.mkdir(exist_ok=True)

all_results = {}
for variant, model, trainer in [
    ('v1', model_v1, trainer_v1),
    ('v2', model_v2, trainer_v2),
    ('v3', model_v3, trainer_v3),
]:
    print(f'\n=== Evaluating {variant.upper()} ===')
    # Load best checkpoint
    ckpt = torch.load(f'../runs/{variant}/best.pt', map_location=DEVICE)
    model.load_state_dict(ckpt['model_state'])
    model = model.to(DEVICE).eval()

    metrics = compute_all_metrics(model, test_loader, variant, DEVICE, stats)
    all_results[variant] = metrics
    save_results_csv(metrics, f'se3_cfm_{variant}', RESULTS_CSV)
    print(f'  Pearson r (mean): {metrics["pearson"]["mean"]:.4f}')
    print(f'  JSD inv (mean):   {metrics["jsd_invariants"]["mean"]:.5f}')
    print(f'  Diss corr:        {metrics["dissipation"]["correlation"]:.4f}')
    print(f'  Align angle:      {metrics["alignment"]["mean_deg"]:.2f} deg')

## 7. Deeper Analysis — Rotational Equivariance Audit

In [ ]:
from evaluation.audit import run_equivariance_audit

# Load best checkpoints for all three
models_for_audit = {}
for variant, model in [('v1', model_v1), ('v2', model_v2), ('v3', model_v3)]:
    ckpt = torch.load(f'../runs/{variant}/best.pt', map_location=DEVICE)
    model.load_state_dict(ckpt['model_state'])
    models_for_audit[variant] = model.to(DEVICE).eval()

audit_results = run_equivariance_audit(
    models_for_audit, test_loader, DEVICE,
    n_subcubes=50, n_rotations=100,
    out_dir='../figures',
)

print('\nEquivariance Audit Results:')
for v, r in audit_results.items():
    print(f'  {v.upper()}: mean={r["mean"]:.4e}  std={r["std"]:.4e}')

print('\nInterpretation:')
print('  V1 (SE3-equivariant) should show machine-precision error (~1e-6).')
print('  V3 (augmentation only) should show higher error for arbitrary rotations.')

## 8. V4 — Unified Results Table (V1 vs all Task 2 baselines)

In [ ]:
from evaluation.metrics import load_results_csv

# Load all results (Task 2 baselines should already be written to this CSV by Sudip)
all_csv = load_results_csv(RESULTS_CSV)

if not all_csv:
    print('No results CSV found yet — run Sudip Task 2 notebook first to populate it.')
else:
    # Print unified results table
    models = sorted(all_csv.keys())
    metrics_to_show = [
        'pearson.mean', 'jsd_invariants.mean',
        'dissipation.correlation', 'alignment.mean_deg',
        'backscatter_fraction', 'efficiency.ms_per_subcube',
    ]

    header = f'{"Model":<30}' + ''.join(f'{m:<22}' for m in metrics_to_show)
    print('\n' + '='*120)
    print('UNIFIED RESULTS TABLE (V4)')
    print('='*120)
    print(header)
    print('-'*120)
    for name in models:
        row = f'{name:<30}'
        for m in metrics_to_show:
            val = all_csv[name].get(m, float('nan'))
            row += f'{val:<22.4f}' if isinstance(val, float) else f'{str(val):<22}'
        print(row)
    print('='*120)

## 9. Uncertainty Quantification (V1 — generative model)

In [ ]:
# Draw multiple samples from V1 for the 5 most uncertain test sub-cubes
import torch
from torch_geometric.data import Batch
from data.dataset import irreps_to_stress

model_v1.eval()
N_SAMPLES = 10

# Collect first 20 test graphs
test_graphs = []
for batch in test_loader:
    test_graphs.extend(batch.to_data_list())
    if len(test_graphs) >= 20:
        break
test_graphs = test_graphs[:20]

stds, trues = [], []
with torch.no_grad():
    for g in test_graphs:
        b = Batch.from_data_list([g]).to(DEVICE)
        samples = torch.stack([model_v1.sample(b, method='euler') for _ in range(N_SAMPLES)])
        # Centre node (index 0 of the sub-cube)
        std = samples[:, 0, :].std(dim=0)   # std over samples, shape [6]
        stds.append(std.cpu().numpy())
        trues.append(g.y[0].numpy())

stds  = np.array(stds)   # [20, 6]
# Identify 5 most uncertain
uncertainty = stds.mean(axis=1)   # [20]
top5 = np.argsort(uncertainty)[-5:][::-1]

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
comp_names = ['l0', 'l2_xy', 'l2_yz', 'l2_m0', 'l2_xz', 'l2_diag']
for ax, idx in zip(axes, top5):
    ax.bar(range(6), stds[idx], color='steelblue', alpha=0.7)
    ax.set_xticks(range(6)); ax.set_xticklabels(comp_names, rotation=45, fontsize=8)
    ax.set_title(f'Sub-cube {idx}\nσ={uncertainty[idx]:.4f}', fontsize=9)
    ax.set_ylabel('Std across 10 samples')
fig.suptitle('V1 prediction uncertainty (most uncertain sub-cubes)', fontsize=11)
plt.tight_layout()
plt.savefig('../figures/v1_uncertainty.png', dpi=150)
plt.show()

## 10. Summary

Key numbers for the paper:
- **V1 SE(3)-CFM**: best Pearson r and JSD; meaningful uncertainty estimates
- **V2 SE(3)-regression**: competitive Pearson r, zero uncertainty
- **V3 MLP-CFM**: degraded equivariance error, lower correlation at arbitrary rotations
- **Equivariance audit**: V1 ≈ machine precision; V3 degrades with rotation angle


In [ ]:
print('=== Task 1 key numbers ===')
for variant in ['v1', 'v2', 'v3']:
    if variant in all_results:
        m = all_results[variant]
        print(f'{variant.upper()}: Pearson={m["pearson"]["mean"]:.4f}  '
              f'JSD={m["jsd_invariants"]["mean"]:.5f}  '
              f'Align={m["alignment"]["mean_deg"]:.2f}°')
print('\nEquivariance errors:')
for v, r in audit_results.items():
    print(f'  {v.upper()}: {r["mean"]:.2e}')